# 07 · BERTopic (transformer embeddings + UMAP + HDBSCAN)
The project doc asks for BERTopic. This notebook runs a **guided** BERTopic (seeded with the same words as notebook 06)
over the same sentences, sweeps a small configuration grid, scores each with NPMI coherence, and writes
`sentence_topics_bertopic.csv` in the same schema as the seeded topics, so notebooks 08/09 can use either.

> ⚠️ **Environment:** needs `bertopic sentence-transformers umap-learn hdbscan` and internet to download the embedding model.
> If unavailable, the notebook skips the heavy cells. The code was **not executed in the authoring sandbox** — expect small
> fixes on first run (BERTopic/UMAP versions differ).
>
> Fitting is done on the **unique** masked sentences (the corpus is highly repetitive), then every sentence is mapped to its topic.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import itertools, json, re
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "outputs"; TOP = OUT / "topics"
SEED = 42; GENERAL = "General / Overall"
MLFLOW_URI = f"sqlite:///{ROOT / 'mlflow.db'}"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
MIN_TOPIC_SIZE = 30

try:
    import mlflow
    from bertopic import BERTopic
    from hdbscan import HDBSCAN
    from sentence_transformers import SentenceTransformer
    from umap import UMAP
    HAVE_BT = True
except Exception as e:                                   # noqa: BLE001
    HAVE_BT = False
    print("⏭  BERTopic cells will be skipped — missing dependency:", repr(e)[:120])

need = [TOP / "sentences.csv", TOP / "seeds.json"]
if not all(p.exists() for p in need):
    HAVE_BT = False; print("⏭  run notebook 06 first (sentences.csv / seeds.json missing)")

⏭  BERTopic cells will be skipped — missing dependency: ModuleNotFoundError("No module named 'bertopic'")


## Inputs and seeds

In [2]:
if HAVE_BT:
    sent = pd.read_csv(TOP / "sentences.csv")
    SEEDS = json.loads((TOP / "seeds.json").read_text())
    SEED_RX = {c: [re.compile(p, re.I) for p in ps] for c, ps in SEEDS.items()}
    SEED_WORDS = {   # plain words for guided BERTopic
        "Staff": ["staff", "receptionist", "concierge", "manager", "team", "polite"],
        "Service": ["check-in", "arrival", "service", "queue", "waited", "keys"],
        "Cleanliness": ["clean", "spotless", "dust", "tidy", "housekeeping", "dirty"],
        "Facilities": ["spa", "gym", "pool", "fitness", "wifi", "lounge", "parking"],
        "Food": ["breakfast", "buffet", "restaurant", "cafe", "dining", "pastries"],
        "Location": ["location", "neighborhood", "transport", "central", "nearby", "convenient"],
        "Room quality": ["room", "bathroom", "bedding", "carpet", "decor", "suite"],
        "Pricing": ["price", "cost", "rate", "value", "overpriced", "bargain"],
        "Comfort": ["comfortable", "mattress", "pillows", "noise", "quiet", "spacious"]}
    uniq = sent.masked.drop_duplicates().tolist()
    print(f"{len(sent):,} sentences, {len(uniq):,} unique")
    emb = SentenceTransformer(EMBEDDING_MODEL).encode(uniq, show_progress_bar=True, batch_size=128)
    cv = CountVectorizer(min_df=3, stop_words="english", binary=True)
    B = cv.fit_transform(uniq); vindex = {t: i for i, t in enumerate(cv.get_feature_names_out())}

    def npmi(topic_terms):
        N = B.shape[0]; dfc = np.asarray(B.sum(0)).ravel(); res = []
        for terms in topic_terms:
            idx = [vindex[t] for t in terms if t in vindex]
            if len(idx) < 2: res.append(np.nan); continue
            sub = B[:, idx]; co = (sub.T @ sub).toarray() / N; p = dfc[idx] / N; v = []
            for i in range(len(idx)):
                for j in range(i + 1, len(idx)):
                    pij = co[i, j]; v.append(-1.0 if pij <= 0 else np.log(pij / (p[i] * p[j])) / -np.log(pij))
            res.append(float(np.mean(v)))
        return res

## Configuration sweep (logged to MLflow)

In [3]:
if HAVE_BT:
    mlflow.set_tracking_uri(MLFLOW_URI); mlflow.set_experiment("guest-topic-modelling")
    results, best = [], None
    for nn, mcs in itertools.product([10, 15, 30], [MIN_TOPIC_SIZE, 2 * MIN_TOPIC_SIZE]):
        umap = UMAP(n_neighbors=nn, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED)
        hdb = HDBSCAN(min_cluster_size=mcs, metric="euclidean", cluster_selection_method="eom", prediction_data=True)
        tm = BERTopic(embedding_model=None, umap_model=umap, hdbscan_model=hdb,
                      vectorizer_model=CountVectorizer(stop_words="english", min_df=3),
                      seed_topic_list=list(SEED_WORDS.values()), calculate_probabilities=False, verbose=False)
        topics, _ = tm.fit_transform(uniq, embeddings=emb)
        info = tm.get_topic_info(); info = info[info.Topic != -1]
        tops = [[w for w, _ in tm.get_topic(t)][:10] for t in info.Topic]
        coh = float(np.nanmean(npmi(tops))) if tops else float("nan"); outl = float(np.mean(np.array(topics) == -1))
        with mlflow.start_run(run_name=f"bertopic_nn{nn}_mcs{mcs}"):
            mlflow.log_params({"method": "bertopic_guided", "embedding": EMBEDDING_MODEL, "n_neighbors": nn, "min_cluster_size": mcs})
            mlflow.log_metrics({"npmi_coherence_mean": coh, "n_topics": len(tops), "outlier_share": outl})
        results.append({"n_neighbors": nn, "min_cluster_size": mcs, "n_topics": len(tops), "npmi": coh, "outlier_share": outl})
        if len(tops) >= 6 and (best is None or coh > best[0]): best = (coh, tm, topics, nn, mcs)
    res = pd.DataFrame(results); print(res.round(3).to_string(index=False))

## Best configuration → map topics to business categories
Each BERTopic topic is mapped to the category whose seed regexes match most of its top words; unmatched topics keep their
BERTopic label; outliers (−1) become *General / Overall*.

In [4]:
if HAVE_BT and best is not None:
    coh, tm, topics, nn, mcs = best
    print(f"best: n_neighbors={nn}, min_cluster_size={mcs}, coherence={coh:.3f}")
    def map_to_category(words):
        text = " ".join(words); hits = {c: sum(len(rx.findall(text)) for rx in rxs) for c, rxs in SEED_RX.items()}
        b = max(hits, key=hits.get); return b if hits[b] > 0 else None
    t_of = dict(zip(uniq, topics)); label_of, rows = {}, []
    for t in tm.get_topic_info().Topic:
        words = [w for w, _ in tm.get_topic(t)][:10] if t != -1 else []
        cat = GENERAL if t == -1 else (map_to_category(words) or f"Topic {t}: " + ", ".join(words[:3]))
        label_of[t] = cat; rows.append({"topic": t, "category": cat, "top_terms": ", ".join(words)})
    topic_info = pd.DataFrame(rows); topic_info.to_csv(TOP / "topic_info_bertopic.csv", index=False)
    sent["category"] = sent.masked.map(t_of).map(label_of)
    sent[["review_id", "sent_idx", "sentence", "category"]].assign(hits=1).to_csv(TOP / "sentence_topics_bertopic.csv", index=False)
    display(topic_info)
    sent.category.value_counts().sort_values().plot.barh(figsize=(8, 4), color="#4C78A8", title="Sentences per BERTopic category")
    plt.tight_layout(); plt.show()

## Using BERTopic downstream
To feed the topic-sentiment notebook (08) and dashboard (09) with BERTopic topics, set `TOPIC_FILE = "sentence_topics_bertopic.csv"`
in notebook 08. Compare against the seeded version in 06 (coverage, interpretability, coherence) before choosing —
on short, template-like review sentences the seeded approach is hard to beat; BERTopic earns its keep on free-form text.